# 08 · NASA Worldview, imagery provenance, and change


**Mission (35 minutes):** connect a dated NASA Worldview view to an auditable imagery comparison. NASA **Worldview** is a separate NASA application; **God's Eye View** is Bilawal Sidhu's situational-awareness globe. Use the former to inspect dated Earth-observation context and the latter to combine scene layers.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

## 1. Construct a dated view, not a claim of an event
The example date is a reproducible archive request, unrelated to the fictional September 2026 exercise. The link opens a real service; network access and product availability are required. Cloud and missing coverage remain possible.

In [ ]:
from urllib.parse import urlencode
imagery_date='2024-09-01'
bbox=[32.4,7.2,33.7,8.4]  # west, south, east, north
worldview='https://worldview.earthdata.nasa.gov/?'+urlencode({'v':','.join(map(str,bbox)),'t':imagery_date,'l':'MODIS_Terra_CorrectedReflectance_TrueColor'})
display(Markdown(f'[Open dated NASA Worldview view]({worldview})'))
# WMS 1.1.1 with EPSG:4326 uses longitude,latitude bounding-box order.
gibs='https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi?'+urlencode({
    'SERVICE':'WMS','REQUEST':'GetMap','VERSION':'1.1.1','LAYERS':'MODIS_Terra_CorrectedReflectance_TrueColor',
    'STYLES':'','FORMAT':'image/png','SRS':'EPSG:4326','BBOX':','.join(map(str,bbox)),
    'WIDTH':1000,'HEIGHT':800,'TIME':imagery_date})
display(Markdown(f'[Request NASA GIBS browse image]({gibs})'))
manifest={'provider':'NASA EOSDIS GIBS','product':'MODIS_Terra_CorrectedReflectance_TrueColor',
          'requested_date':imagery_date,'bbox_crs84':bbox,'request_url':gibs,
          'acquisition_time':'VERIFY IN PRODUCT METADATA','cloud_quality':'NOT ASSESSED',
          'interpretation':'browse visualization; not a calibrated reflectance raster'}
download('imagery-manifest.json',manifest)

## 2. Learn change detection on explicit synthetic spectral arrays
RGB screenshots are not calibrated spectral bands. Here we construct invented green/NIR reflectance arrays on a local **1 km grid** and compute NDWI = (green − NIR)/(green + NIR). Thresholding is a toy method; a cloud mask is essential. Never compute this spectral index from a colored map screenshot.

In [ ]:
y,x=np.mgrid[:60,:80]; axis=35+7*np.sin(y/10)
before_water=np.abs(x-axis)<3
after_water=np.abs(x-axis)<(3+8*np.exp(-((y-30)/13)**2))
cloud=(x-59)**2+(y-20)**2 < 100
def synthetic_ndwi(water):
    green=np.where(water,.30,.16);nir=np.where(water,.06,.38)
    return (green-nir)/(green+nir)
before=synthetic_ndwi(before_water);after=synthetic_ndwi(after_water)
after[cloud]=np.nan
change=(after>0)&~(before>0)&np.isfinite(after)
fig,axes=plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
observed_change=np.where(np.isfinite(after),change.astype(float),np.nan)
for ax,arr,title in zip(axes,[before,after,observed_change],['Before: synthetic NDWI','After: cloud masked','New water-like pixels']):
    im=ax.imshow(arr,origin='lower',cmap='BrBG',vmin=-1 if 'NDWI' in title or 'cloud' in title else 0,vmax=1,extent=[0,80,0,60]);ax.set(title=title,xlabel='Local x (km)',ylabel='Local y (km)')
    fig.colorbar(im,ax=ax,shrink=.7,label='new water mask (0/1)' if 'pixels' in title else 'NDWI (unitless)')
plt.show()
print('New water-like area among observed pixels:',int(change.sum()),'km² (synthetic 1 km² cells)')
print('Cloud/missing area:',int(cloud.sum()),'km²; excluded, not dry')

## Lab challenge and Astra prompt
Change the threshold from 0 to 0.2, then expand the cloud mask. Explain why the observed area is not the total flooded area. Outline validation with a product quality layer, dates, sensor resolution, and independent field observations.

**Image-grounded prompt:** “Using this exported map, its legend, and the attached manifest, separate visible observations from hypotheses. Cite the layer/date for each observation. State what cloud and resolution prevent you from concluding. Do not infer disease, identify people, or estimate precise area from screenshot pixels.” In notebook 10, numerical evidence comes from computed arrays; Astra can help explain and critique it.

**Sources:** [NASA GIBS access guide](https://nasa-gibs.github.io/gibs-api-docs/access-basics/), [NASA Worldview source](https://github.com/nasa-gibs/worldview), [OpenAI image-input limitations](https://developers.openai.com/api/docs/guides/images-vision). No NASA imagery is bundled.